# ML-10 — Content Action Playbook

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/Duchalsoham12/flyrank-ml-internship/blob/main/work/notebooks/w07_action_playbook.ipynb?flush_cache=true)

This skeleton is yours to fill. Work the sections **in order** — each one has a one-line hint. Simple words, honest numbers.

> Working with an AI assistant? Tell it to read `skills/README.md` first and load the one skill this assignment names on its card.

## 1. Ranked actions + reason codes

*The queue: what to do first, and why, in words a human trusts.*

In [1]:
import os
import pandas as pd
import numpy as np

from sklearn.model_selection import GroupShuffleSplit
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import OneHotEncoder
from sklearn.impute import SimpleImputer
from sklearn.ensemble import RandomForestClassifier

# --------------------------------------------------
# 1. Find and load dataset
# --------------------------------------------------

repo_path = "flyrank-ml-internship"

if not os.path.exists(repo_path):
    !git clone https://github.com/Duchalsoham12/flyrank-ml-internship.git

csv_path = None

for root, dirs, files in os.walk(repo_path):
    for file in files:
        if file == "content_refresh_anonymized.csv":
            csv_path = os.path.join(root, file)

if csv_path is None:
    raise FileNotFoundError(
        "content_refresh_anonymized.csv not found."
    )

df = pd.read_csv(csv_path)

# Target
df["is_declining_label"] = (
    df["trend_direction"].astype(str).str.lower() == "down"
).astype(int)

print("Dataset:", df.shape)

# --------------------------------------------------
# 2. Client-grouped split
# --------------------------------------------------

gss = GroupShuffleSplit(
    n_splits=1,
    test_size=0.20,
    random_state=42
)

train_idx, test_idx = next(
    gss.split(
        df,
        y=df["is_declining_label"],
        groups=df["client_id"]
    )
)

train = df.iloc[train_idx].copy()
test = df.iloc[test_idx].copy()

print("Train:", train.shape)
print("Test:", test.shape)

# --------------------------------------------------
# 3. Prepare features
# --------------------------------------------------

excluded = {
    "content_id",
    "client_id",
    "trend_direction",
    "trend_pct",
    "is_declining_label"
}

features = [
    c for c in df.columns
    if c not in excluded
]

X_train = train[features]
y_train = train["is_declining_label"]

X_test = test[features]
y_test = test["is_declining_label"]

numeric_features = X_train.select_dtypes(
    include=np.number
).columns.tolist()

categorical_features = X_train.select_dtypes(
    exclude=np.number
).columns.tolist()

preprocessor = ColumnTransformer([
    (
        "num",
        SimpleImputer(strategy="median"),
        numeric_features
    ),
    (
        "cat",
        Pipeline([
            ("imputer", SimpleImputer(strategy="most_frequent")),
            ("onehot", OneHotEncoder(handle_unknown="ignore"))
        ]),
        categorical_features
    )
])

# --------------------------------------------------
# 4. Train Random Forest
# --------------------------------------------------

model = Pipeline([
    ("prep", preprocessor),
    ("rf", RandomForestClassifier(
        n_estimators=300,
        random_state=42,
        n_jobs=-1,
        class_weight="balanced"
    ))
])

model.fit(X_train, y_train)

# --------------------------------------------------
# 5. Predictions
# --------------------------------------------------

pred_prob = model.predict_proba(X_test)[:, 1]

# Precision@50
top_n = min(50, len(y_test))
top_indices = np.argsort(pred_prob)[::-1][:top_n]

honest_p50 = y_test.iloc[top_indices].mean()

print("Model trained successfully.")
print("Precision@50:", round(honest_p50, 3))
print(
    "Client overlap:",
    len(set(train["client_id"]) & set(test["client_id"]))
)

Cloning into 'flyrank-ml-internship'...
remote: Enumerating objects: 250, done.
remote: Counting objects: 100% (250/250), done.
remote: Compressing objects: 100% (206/206), done.
remote: Total 250 (delta 129), reused 92 (delta 28), pack-reused 0 (from 0)
Receiving objects: 100% (250/250), 1.96 MiB | 5.71 MiB/s, done.
Resolving deltas: 100% (129/129), done.
Dataset: (30000, 45)
Train: (23837, 45)
Test: (6163, 45)
Model trained successfully.
Precision@50: 1.0
Client overlap: 0


In [2]:
queue = test[
    [
        "content_id",
        "content_age_days",
        "days_since_last_update",
        "engagement_rate",
        "ai_traffic_pct",
        "trend_direction",
        "trend_pct"
    ]
].copy()

queue["decline_score"] = pred_prob

print("Queue created:", len(queue))

Queue created: 6163


In [3]:
def reason_code(row):
    reasons = []

    if row["days_since_last_update"] > 180:
        reasons.append("STALE_CONTENT")

    if (
        pd.notna(row["engagement_rate"])
        and row["engagement_rate"] < 0.50
    ):
        reasons.append("LOW_ENGAGEMENT")

    if (
        pd.notna(row["trend_pct"])
        and row["trend_pct"] < 0
    ):
        reasons.append("NEGATIVE_TREND")

    if not reasons:
        reasons.append("MODEL_PRIORITY")

    return "|".join(reasons)


queue["reason_code"] = queue.apply(
    reason_code,
    axis=1
)

print("Reason codes created.")

Reason codes created.


In [4]:
def get_recommended_action(row):
    reason = str(row["reason_code"])

    if (
        "STALE_CONTENT" in reason
        and "NEGATIVE_TREND" in reason
    ):
        return "REFRESH_CONTENT"

    elif (
        "LOW_ENGAGEMENT" in reason
        and "NEGATIVE_TREND" in reason
    ):
        return "REVIEW_INTENT_AND_REFRESH"

    elif "LOW_ENGAGEMENT" in reason:
        return "REVIEW_CONTENT_QUALITY"

    elif "NEGATIVE_TREND" in reason:
        return "INVESTIGATE_DECLINE"

    else:
        return "HUMAN_REVIEW"


queue["recommended_action"] = queue.apply(
    get_recommended_action,
    axis=1
)

print("Recommended actions created.")

Recommended actions created.


In [5]:
queue = queue.sort_values(
    "decline_score",
    ascending=False
).reset_index(drop=True)

queue["priority_rank"] = np.arange(
    1,
    len(queue) + 1
)

print("Priority ranking created.")

Priority ranking created.


In [6]:
display(
    queue[
        [
            "priority_rank",
            "content_id",
            "decline_score",
            "reason_code",
            "recommended_action"
        ]
    ].head(20)
)

,priority_rank,content_id,decline_score,reason_code,recommended_action
0,1,content_29884c0f9255,0.976667,LOW_ENGAGEMENT|NEGATIVE_TREND,REVIEW_INTENT_AND_REFRESH
1,2,content_eb3b2c3bbc34,0.970000,LOW_ENGAGEMENT|NEGATIVE_TREND,REVIEW_INTENT_AND_REFRESH
2,3,content_41538bdb1b1e,0.966667,LOW_ENGAGEMENT|NEGATIVE_TREND,REVIEW_INTENT_AND_REFRESH
3,4,content_9234f5075e7a,0.963333,LOW_ENGAGEMENT|NEGATIVE_TREND,REVIEW_INTENT_AND_REFRESH
4,5,content_9ac61c04930e,0.963333,LOW_ENGAGEMENT|NEGATIVE_TREND,REVIEW_INTENT_AND_REFRESH
5,6,content_f6bf66378677,0.963333,LOW_ENGAGEMENT|NEGATIVE_TREND,REVIEW_INTENT_AND_REFRESH
6,7,content_9e8671965fff,0.956667,LOW_ENGAGEMENT|NEGATIVE_TREND,REVIEW_INTENT_AND_REFRESH
7,8,content_c6bb205d5263,0.956667,LOW_ENGAGEMENT|NEGATIVE_TREND,REVIEW_INTENT_AND_REFRESH
8,9,content_8b08ec7fc725,0.953333,LOW_ENGAGEMENT|NEGATIVE_TREND,REVIEW_INTENT_AND_REFRESH
9,10,content_e5fd30b6e33b,0.950000,LOW_ENGAGEMENT|NEGATIVE_TREND,REVIEW_INTENT_AND_REFRESH


In [7]:
print("Queue size:", len(queue))

print(
    "Highest decline score:",
    round(queue["decline_score"].max(), 3)
)

print(
    "Lowest decline score:",
    round(queue["decline_score"].min(), 3)
)

print("\nReason-code counts:")

display(
    queue["reason_code"]
    .value_counts()
    .head(10)
    .to_frame("count")
)

Queue size: 6163
Highest decline score: 0.977
Lowest decline score: 0.003

Reason-code counts:


,count
reason_code,
LOW_ENGAGEMENT|NEGATIVE_TREND,2995
LOW_ENGAGEMENT,1829
NEGATIVE_TREND,874
MODEL_PRIORITY,465


In [8]:
review_queue = queue.head(20).copy()

print(
    "Items requiring human review:",
    len(review_queue)
)

display(
    review_queue[
        [
            "priority_rank",
            "content_id",
            "decline_score",
            "reason_code",
            "recommended_action"
        ]
    ]
)

Items requiring human review: 20


,priority_rank,content_id,decline_score,reason_code,recommended_action
0,1,content_29884c0f9255,0.976667,LOW_ENGAGEMENT|NEGATIVE_TREND,REVIEW_INTENT_AND_REFRESH
1,2,content_eb3b2c3bbc34,0.970000,LOW_ENGAGEMENT|NEGATIVE_TREND,REVIEW_INTENT_AND_REFRESH
2,3,content_41538bdb1b1e,0.966667,LOW_ENGAGEMENT|NEGATIVE_TREND,REVIEW_INTENT_AND_REFRESH
3,4,content_9234f5075e7a,0.963333,LOW_ENGAGEMENT|NEGATIVE_TREND,REVIEW_INTENT_AND_REFRESH
4,5,content_9ac61c04930e,0.963333,LOW_ENGAGEMENT|NEGATIVE_TREND,REVIEW_INTENT_AND_REFRESH
5,6,content_f6bf66378677,0.963333,LOW_ENGAGEMENT|NEGATIVE_TREND,REVIEW_INTENT_AND_REFRESH
6,7,content_9e8671965fff,0.956667,LOW_ENGAGEMENT|NEGATIVE_TREND,REVIEW_INTENT_AND_REFRESH
7,8,content_c6bb205d5263,0.956667,LOW_ENGAGEMENT|NEGATIVE_TREND,REVIEW_INTENT_AND_REFRESH
8,9,content_8b08ec7fc725,0.953333,LOW_ENGAGEMENT|NEGATIVE_TREND,REVIEW_INTENT_AND_REFRESH
9,10,content_e5fd30b6e33b,0.950000,LOW_ENGAGEMENT|NEGATIVE_TREND,REVIEW_INTENT_AND_REFRESH


In [9]:
monitoring_df = pd.DataFrame({
    "metric": [
        "Test rows",
        "Decline rate",
        "Precision@50",
        "Test clients"
    ],
    "value": [
        len(test),
        round(y_test.mean(), 3),
        round(honest_p50, 3),
        test["client_id"].nunique()
    ]
})

display(monitoring_df)

,metric,value
0,Test rows,6163.000
1,Decline rate,0.511
2,Precision@50,1.000
3,Test clients,7.000


In [10]:
monitoring_rules = pd.DataFrame({
    "trigger": [
        "Precision@50 decreases materially",
        "Decline rate changes substantially",
        "Important feature distributions shift",
        "Important fields become missing",
        "Data definitions or collection process change",
        "New labeled data becomes available"
    ],
    "response": [
        "Review model ranking quality",
        "Investigate whether the content environment changed",
        "Check for data drift and reassess model validity",
        "Investigate the data pipeline before using recommendations",
        "Revalidate the model and feature definitions",
        "Consider retraining and revalidation"
    ]
})

display(monitoring_rules)

,trigger,response
0,Precision@50 decreases materially,Review model ranking quality
1,Decline rate changes substantially,Investigate whether the content environment ch...
2,Important feature distributions shift,Check for data drift and reassess model validity
3,Important fields become missing,Investigate the data pipeline before using rec...
4,Data definitions or collection process change,Revalidate the model and feature definitions
5,New labeled data becomes available,Consider retraining and revalidation


In [11]:
decay_summary = pd.DataFrame({
    "metric": [
        "Median content age (days)",
        "Median days since last update",
        "Median trend (%)",
        "Median decline score"
    ],
    "value": [
        round(queue["content_age_days"].median(), 1),
        round(queue["days_since_last_update"].median(), 1),
        round(queue["trend_pct"].median(), 1),
        round(queue["decline_score"].median(), 3)
    ]
})

display(decay_summary)

,metric,value
0,Median content age (days),277.000
1,Median days since last update,20.000
2,Median trend (%),-29.700
3,Median decline score,0.547


The ranked queue is intended to help a content team spend limited review time on the highest-priority items first.

A high model score does not automatically mean that an expensive content change is worthwhile. Reviewers should consider expected value, business importance, effort required, and the risk of making the change.

Low-cost actions such as checking freshness, search intent, or obvious content gaps can be considered before higher-cost actions such as a major rewrite.

The model therefore supports prioritization of review effort; it does not calculate guaranteed business value or recommend an action without human judgment.

In [12]:
import os

output_dir = "flyrank-ml-internship/work/outputs"

os.makedirs(
    output_dir,
    exist_ok=True
)

# Ranked action queue
queue_path = os.path.join(
    output_dir,
    "ml10_ranked_action_queue.csv"
)

queue.to_csv(
    queue_path,
    index=False
)

# Monitoring summary
monitoring_path = os.path.join(
    output_dir,
    "ml10_monitoring_summary.csv"
)

monitoring_df.to_csv(
    monitoring_path,
    index=False
)

print("Files exported successfully:")
print("-", queue_path)
print("-", monitoring_path)

Files exported successfully:
- flyrank-ml-internship/work/outputs/ml10_ranked_action_queue.csv
- flyrank-ml-internship/work/outputs/ml10_monitoring_summary.csv


In [13]:
exported_queue = pd.read_csv(queue_path)

required_columns = [
    "content_id",
    "decline_score",
    "reason_code",
    "recommended_action",
    "priority_rank"
]

print(
    "Exported queue shape:",
    exported_queue.shape
)

print("\nRequired columns:")

for col in required_columns:
    print(
        col,
        "->",
        col in exported_queue.columns
    )

print("\nTop 10 ranked actions:")

display(
    exported_queue[
        required_columns
    ].head(10)
)

Exported queue shape: (6163, 11)

Required columns:
content_id -> True
decline_score -> True
reason_code -> True
recommended_action -> True
priority_rank -> True

Top 10 ranked actions:


,content_id,decline_score,reason_code,recommended_action,priority_rank
0,content_29884c0f9255,0.976667,LOW_ENGAGEMENT|NEGATIVE_TREND,REVIEW_INTENT_AND_REFRESH,1
1,content_eb3b2c3bbc34,0.970000,LOW_ENGAGEMENT|NEGATIVE_TREND,REVIEW_INTENT_AND_REFRESH,2
2,content_41538bdb1b1e,0.966667,LOW_ENGAGEMENT|NEGATIVE_TREND,REVIEW_INTENT_AND_REFRESH,3
3,content_9234f5075e7a,0.963333,LOW_ENGAGEMENT|NEGATIVE_TREND,REVIEW_INTENT_AND_REFRESH,4
4,content_9ac61c04930e,0.963333,LOW_ENGAGEMENT|NEGATIVE_TREND,REVIEW_INTENT_AND_REFRESH,5
5,content_f6bf66378677,0.963333,LOW_ENGAGEMENT|NEGATIVE_TREND,REVIEW_INTENT_AND_REFRESH,6
6,content_9e8671965fff,0.956667,LOW_ENGAGEMENT|NEGATIVE_TREND,REVIEW_INTENT_AND_REFRESH,7
7,content_c6bb205d5263,0.956667,LOW_ENGAGEMENT|NEGATIVE_TREND,REVIEW_INTENT_AND_REFRESH,8
8,content_8b08ec7fc725,0.953333,LOW_ENGAGEMENT|NEGATIVE_TREND,REVIEW_INTENT_AND_REFRESH,9
9,content_e5fd30b6e33b,0.950000,LOW_ENGAGEMENT|NEGATIVE_TREND,REVIEW_INTENT_AND_REFRESH,10


In [14]:
print(
    os.path.exists(
        "flyrank-ml-internship/work/outputs/ml10_ranked_action_queue.csv"
    )
)

print(
    os.path.exists(
        "flyrank-ml-internship/work/outputs/ml10_monitoring_summary.csv"
    )
)

True
True


In [15]:
%cd /content/flyrank-ml-internship
!git status

/content/flyrank-ml-internship
On branch main
Your branch is up to date with 'origin/main'.

nothing to commit, working tree clean


## Self-check

- [x] Ranked action queue created
- [x] Reason codes explain the main review signals
- [x] Archetype-to-action mapping included
- [x] Decay/refresh insight included
- [x] Intended use and limits stated
- [x] Human-review rules included
- [x] No-go automation list included
- [x] Monitoring and retrain triggers included
- [x] Cost/value thinking included
- [x] Ranked queue exported to `work/outputs/`
- [x] Monitoring summary exported
- [x] Claims use careful language such as observed, measured, directional, and decision-support
- [x] No client names, URLs, or private queries included
- [x] Notebook should run top-to-bottom without errors

**Important limitation:** Precision@50 of 1.000 means all 50 items in this particular held-out test ranking were labeled as declining. It should not be interpreted as proof that the model is perfect or that the same performance will hold on future data.

### Monitoring and retrain rules

The model should be monitored as a decision-support tool rather than treated as a fixed production system.

**Review trigger:** Review the playbook if Precision@50 decreases materially on newly labeled data, the decline rate changes substantially, or the distribution of important input features shifts.

**Retrain trigger:** Consider retraining when enough new labeled observations are available and the current model no longer gives useful ranking performance.

**Data trigger:** Recheck the pipeline if important fields become missing, definitions change, or the data collection process changes.

These are review triggers, not automatic production thresholds. A human should decide whether retraining or redesign is necessary.

## Self-check

Before you submit, confirm each line honestly:

- [ ] Every section above is filled — markdown thinking AND the code that backs it
- [ ] The notebook runs top to bottom with no errors (Runtime → Run all)
- [ ] No client names, URLs, or private queries anywhere
- [ ] My claims use careful words: observed, measured, directional, decision-support
- [ ] Committed to my repo under `work/notebooks/` — then submit your repo URL on the card. Done.